### Install TensorFlow library

In [ ]:
!pip install tensorflow

### Import libraries for image processing, visualization, and building the transfer learning model

In [ ]:
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import applications
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

### Unzip dataset from Google Drive

In [ ]:
!unzip /content/drive/MyDrive/archive.zip

### Define and run a function to scan the dataset and remove any corrupted images

In [ ]:
def clean_corrupted_images(path):
    deleted = 0
    for cls in os.listdir(path):
        p = os.path.join(path, cls)
        if not os.path.isdir(p):
            continue
        for file in os.listdir(p):
            fp = os.path.join(p, file)
            try:
                with Image.open(fp) as img:
                    img.verify()
            except Exception:
                print(f"Deleted corrupted image: {fp}")
                os.remove(fp)
                deleted += 1
    print(f"\nCleaning complete. Deleted {deleted} corrupted images.")

clean_corrupted_images("PetImages")

### Set hyperparameters and configure data generators with augmentation for training and validation splits

In [ ]:
img_size = (224, 224)
batch_size = 32
epochs = 5
folder = "PetImages"
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

train_generator = datagen.flow_from_directory(
    directory=folder,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    directory=folder,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)

### Build the model using MobileNetV2 as a frozen base with a custom classification head

In [ ]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(*img_size, 3))
base_model.trainable = False
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.4),
    Dense(1, activation='sigmoid')
])

model.summary()

### Compile the model with Adam optimizer and binary cross-entropy loss

In [ ]:
model.compile(optimizer=Adam(1e-4),
              loss='binary_crossentropy',
              metrics=['accuracy'])

### Train the model on the training set and evaluate on the validation set each epoch

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=epochs
)

### Save the trained model weights to disk

In [ ]:
model.save_weights("model.weights.h5")